# Data Profiling

In [0]:
RAW_BUCKET = "gs://market-intelligence-raw"
print(RAW_BUCKET)

In [0]:
item_details_df = spark.read.json(
    f"{RAW_BUCKET}/ebay/item_details/*.jsonl.gz"
)

display(item_details_df.limit(5))

In [0]:
browse_df = spark.read.json(
    f"{RAW_BUCKET}/ebay/browse_search/*.jsonl.gz"
)

display(browse_df.limit(5))

**Profiling foundation**

In [0]:
from pyspark.sql import functions as F

# ============================================================
# Dataset references
# ============================================================

DATASETS = {
    "browse_search": browse_df,
    "item_details": item_details_df,
}

for name, df in DATASETS.items():
    print(f"{name}:")
    print(f"  rows    : {df.count():,}")
    print(f"  columns : {len(df.columns):,}")
    print()

**Schema profiling**

In [0]:
def profile_schema(df, dataset_name: str):
    rows = []

    for field in df.schema.fields:
        rows.append(
            (
                field.name,
                field.dataType.simpleString(),
                field.nullable,
            )
        )

    return spark.createDataFrame(
        rows,
        ["column_name", "data_type", "nullable"]
    ).withColumn(
        "dataset",
        F.lit(dataset_name)
    ).select(
        "dataset",
        "column_name",
        "data_type",
        "nullable",
    )

In [0]:
browse_schema = profile_schema(
    browse_df,
    "browse_search"
)

display(browse_schema)

In [0]:
item_details_schema = profile_schema(
    item_details_df,
    "item_details"
)

display(item_details_schema)

**Completeness profiling**

In [0]:
def profile_completeness(df):
    total_rows = df.count()

    expressions = []

    for column_name in df.columns:
        expressions.extend([
            F.count(F.col(column_name)).alias(f"{column_name}__non_null"),
            F.sum(
                F.when(
                    F.col(column_name).isNull(),
                    1
                ).otherwise(0)
            ).alias(f"{column_name}__null"),
        ])

    result = df.agg(*expressions)

    profile_rows = []

    for column_name in df.columns:
        non_null_col = f"{column_name}__non_null"
        null_col = f"{column_name}__null"

        row = result.select(
            F.lit(column_name).alias("column_name"),
            F.col(non_null_col).alias("non_null_count"),
            F.col(null_col).alias("null_count"),
        ).withColumn(
            "total_rows",
            F.lit(total_rows)
        ).withColumn(
            "null_percentage",
            F.round(
                F.col("null_count") / F.col("total_rows") * 100,
                2
            )
        )

        profile_rows.append(row)

    final_df = profile_rows[0]

    for row in profile_rows[1:]:
        final_df = final_df.unionByName(row)

    return final_df.orderBy(
        F.col("null_percentage").desc()
    )

In [0]:
browse_completeness = profile_completeness(browse_df)

display(browse_completeness)

In [0]:
item_details_completeness = profile_completeness(item_details_df)

display(item_details_completeness)